In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/vagishayadav166/sgr-photon-dataset/sgr_photon_dataset.csv


In [3]:
import os
for sub in ["models", "scalers", "data"]:
    os.makedirs(f"/kaggle/working/project/{sub}", exist_ok=True)

In [7]:
"""
Unified pipeline for the SGR photon-geodesic surrogate models.

Produces:
  project/
  ├── models/
  │   ├── rmin_model.keras
  │   ├── phi_model.keras
  │   ├── alpha_model.keras        (= deflection_angle model)
  │   ├── capture_model.keras
  │   └── multitask_model.keras
  ├── scalers/
  │   ├── x_scaler.pkl             (shared, fit ONCE on the unified dataset)
  │   ├── rmin_scaler.pkl
  │   ├── phi_scaler.pkl
  │   └── alpha_scaler.pkl
  └── data/
      └── dataset.csv

Run once, from wherever your dataset.csv lives:
    python build_multitask_pipeline.py --csv /path/to/sgr_photon_dataset.csv

Why this differs from your four original notebooks:
  - Filtering (r_min >= 2M) is applied ONCE, consistently, to the whole
    dataset before any split/scale/train happens. Your rmin/capture
    notebooks filtered; phi/deflection did not. That mismatch is why you
    can't reuse any of the four scalers those notebooks fit as a shared
    x_scaler -- they were fit on different underlying distributions.
  - One train/val/test split is computed once and reused for every head,
    so all five models (4 single-task + multitask) see identical rows.
  - Targets (r_min, phi_final, deflection_angle) are now scaled too,
    which is why rmin_scaler / phi_scaler / alpha_scaler exist as
    separate objects. `captured` is binary and needs no scaler.

v2 additions (phi/alpha were badly underfit near the photon sphere):
  - Added an engineered feature, log_crit_dist = log(|b_over_bcrit-1|+eps),
    to the shared feature set. phi_final and deflection_angle both diverge
    as b_over_bcrit -> 1; this feature is roughly linear in the thing that
    was actually diverging, giving the net an easier function to learn.
    It's added for ALL models (not just phi/alpha) so x_scaler stays a
    single shared scaler valid for every model, including the multitask
    trunk -- rmin/capture can just learn to ignore it if unhelpful.
  - phi_final and deflection_angle are now trained in signed-log space:
    signed_log1p(y) = sign(y) * log1p(|y|), inverted with
    sign(z) * expm1(|z|). phi_scaler / alpha_scaler now fit on the
    transformed target, so inference must inverse-transform with the
    scaler AND then apply inverse_signed_log1p, in that order.
  - capture_model uses a per-sample weight (sqrt of the class imbalance
    ratio) via sample_weight -- this works fine for single-output models.
    The multitask captured head instead bakes the same weight into a
    custom weighted binary-crossentropy loss function, because Keras 3
    has a bug (KeyError: 0 in compile_utils.resolve_path) when
    sample_weight is passed as a dict alongside dict-structured y for a
    functional multi-output model. A per-output custom loss sidesteps
    that broken code path entirely.
"""

import argparse
import os
import random
import shutil

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

FEATURES = ["M", "b", "b_over_bcrit", "log_crit_dist"]
RANDOM_STATE = 42
LOG_CRIT_EPS = 1e-6


def set_seeds(seed=RANDOM_STATE):
    # Without this, weight init / batch order differ run-to-run, so
    # results (esp. for the small imbalanced capture_model) shift a lot
    # between otherwise-identical runs -- makes any tuning unreliable.
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


def signed_log1p(y):
    y = np.asarray(y, dtype=float)
    return np.sign(y) * np.log1p(np.abs(y))


def inverse_signed_log1p(z):
    z = np.asarray(z, dtype=float)
    return np.sign(z) * np.expm1(np.abs(z))


def build_dirs(root):
    for sub in ["models", "scalers", "data"]:
        os.makedirs(os.path.join(root, sub), exist_ok=True)


def load_and_clean(csv_path):
    df = pd.read_csv(csv_path)
    if "n_steps" in df.columns:
        df = df.drop(columns=["n_steps"])

    # Impute missing deflection_angle (same approach as your notebooks,
    # run once instead of four times)
    if df["deflection_angle"].isna().any():
        predictor_cols = ["M", "b", "b_over_bcrit", "captured", "r_min", "phi_final", "n_orbits"]
        known = df[df["deflection_angle"].notna()]
        missing = df[df["deflection_angle"].isna()]
        rf = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE)
        rf.fit(known[predictor_cols], known["deflection_angle"])
        df.loc[df["deflection_angle"].isna(), "deflection_angle"] = rf.predict(missing[predictor_cols])

    # Physical validity filter, applied ONCE, globally.
    # (Your rmin/capture notebooks did this; phi/deflection didn't --
    #  that inconsistency is resolved here so every head trains on the
    #  same rows and the shared x_scaler is valid for all of them.)
    df = df[df["r_min"] >= (2 * df["M"])].reset_index(drop=True)

    # Engineered feature: roughly linear in the quantity that phi_final and
    # deflection_angle actually diverge in (distance from the critical
    # impact parameter), computed purely from inputs so it's valid at
    # inference time too.
    df["log_crit_dist"] = np.log(np.abs(df["b_over_bcrit"] - 1.0) + LOG_CRIT_EPS)

    return df


def split(df):
    X = df[FEATURES]
    y_rmin = df["r_min"]
    y_phi = df["phi_final"]
    y_alpha = df["deflection_angle"]
    y_cap = df["captured"]

    idx = df.index.values
    idx_trainval, idx_test = train_test_split(idx, test_size=0.15, random_state=RANDOM_STATE)
    idx_train, idx_val = train_test_split(idx_trainval, test_size=(0.15 / 0.85), random_state=RANDOM_STATE)

    def sel(i):
        return (X.loc[i], y_rmin.loc[i], y_phi.loc[i], y_alpha.loc[i], y_cap.loc[i])

    return sel(idx_train), sel(idx_val), sel(idx_test)


def make_regressor():
    return models.Sequential([
        layers.Dense(64, activation="relu", input_shape=(len(FEATURES),)),
        layers.Dense(128, activation="relu"),
        layers.Dense(64, activation="relu"),
        layers.Dense(1),
    ])


def make_classifier():
    return models.Sequential([
        layers.Dense(64, activation="relu", input_shape=(len(FEATURES),)),
        layers.Dense(128, activation="relu"),
        layers.Dense(64, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])


def train_single(model, X_tr, y_tr, X_val, y_val, loss, metrics, sample_weight=None, val_sample_weight=None):
    model.compile(loss=loss, optimizer=tf.keras.optimizers.Adam(1e-3), metrics=metrics)
    es = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
    val_data = (X_val, y_val) if val_sample_weight is None else (X_val, y_val, val_sample_weight)
    model.fit(X_tr, y_tr, epochs=100, batch_size=32,
              validation_data=val_data, callbacks=[es], verbose=0,
              sample_weight=sample_weight)
    return model


def make_weighted_bce(pos_weight):
    # Bakes the class-imbalance weighting into the loss itself, instead of
    # going through fit()'s sample_weight/class_weight. Keras 3 has a bug
    # (KeyError: 0 in compile_utils.resolve_path) when sample_weight is
    # passed as a dict alongside dict-structured y for a functional
    # multi-output model -- this happens even with matched keys and even
    # with validation-side weighting removed. A per-output custom loss
    # sidesteps that path entirely.
    pos_weight = float(pos_weight)

    def weighted_bce(y_true, y_pred):
        y_true = tf.cast(y_true, y_pred.dtype)
        bce = tf.keras.backend.binary_crossentropy(y_true, y_pred)
        weight = y_true * pos_weight + (1.0 - y_true)
        return tf.reduce_mean(weight * bce)

    return weighted_bce


def build_multitask_model(cap_pos_weight=1.0):
    inp = layers.Input(shape=(len(FEATURES),), name="x")
    trunk = layers.Dense(64, activation="relu")(inp)
    trunk = layers.Dense(128, activation="relu")(trunk)
    trunk = layers.Dense(64, activation="relu")(trunk)

    rmin_out = layers.Dense(32, activation="relu")(trunk)
    rmin_out = layers.Dense(1, name="rmin")(rmin_out)

    phi_out = layers.Dense(32, activation="relu")(trunk)
    phi_out = layers.Dense(1, name="phi")(phi_out)

    alpha_out = layers.Dense(32, activation="relu")(trunk)
    alpha_out = layers.Dense(1, name="alpha")(alpha_out)

    cap_out = layers.Dense(32, activation="relu")(trunk)
    cap_out = layers.Dense(1, activation="sigmoid", name="captured")(cap_out)

    model = models.Model(inputs=inp, outputs=[rmin_out, phi_out, alpha_out, cap_out], name="multitask_model")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss={"rmin": "mse", "phi": "mse", "alpha": "mse",
              "captured": make_weighted_bce(cap_pos_weight)},
        loss_weights={"rmin": 1.0, "phi": 1.0, "alpha": 1.0, "captured": 1.0},
        metrics={"rmin": "mae", "phi": "mae", "alpha": "mae", "captured": "accuracy"},
    )
    return model


def main(csv_path, out_root):
    set_seeds()
    build_dirs(out_root)

    df = load_and_clean(csv_path)
    shutil.copy(csv_path, os.path.join(out_root, "data", "dataset.csv"))

    (X_tr, ytr_rmin, ytr_phi, ytr_alpha, ytr_cap), \
    (X_val, yval_rmin, yval_phi, yval_alpha, yval_cap), \
    (X_te, yte_rmin, yte_phi, yte_alpha, yte_cap) = split(df)

    # ---- shared X scaler, fit ONCE ----
    x_scaler = StandardScaler().fit(X_tr)
    X_tr_s = x_scaler.transform(X_tr)
    X_val_s = x_scaler.transform(X_val)
    X_te_s = x_scaler.transform(X_te)
    joblib.dump(x_scaler, os.path.join(out_root, "scalers", "x_scaler.pkl"))

    # ---- per-target y scalers ----
    # rmin: plain StandardScaler, it was already ~perfect, don't touch it.
    # phi/alpha: fit the scaler on the SIGNED-LOG of the target, since
    # that's the space the net now trains in.
    rmin_scaler = StandardScaler().fit(ytr_rmin.values.reshape(-1, 1))
    phi_scaler = StandardScaler().fit(signed_log1p(ytr_phi.values).reshape(-1, 1))
    alpha_scaler = StandardScaler().fit(signed_log1p(ytr_alpha.values).reshape(-1, 1))
    joblib.dump(rmin_scaler, os.path.join(out_root, "scalers", "rmin_scaler.pkl"))
    joblib.dump(phi_scaler, os.path.join(out_root, "scalers", "phi_scaler.pkl"))
    joblib.dump(alpha_scaler, os.path.join(out_root, "scalers", "alpha_scaler.pkl"))

    def sc(scaler, y):
        return scaler.transform(y.values.reshape(-1, 1)).ravel()

    def sc_log(scaler, y):
        # phi/alpha targets: signed-log transform, then scale
        return scaler.transform(signed_log1p(y.values).reshape(-1, 1)).ravel()

    def inv_log(scaler, z):
        # inverse of sc_log: unscale, then undo the signed-log transform
        return inverse_signed_log1p(scaler.inverse_transform(z))

    # sample_weight for the capture head: sqrt of the raw imbalance ratio,
    # not the full ratio. The full ratio (~15:1) overcorrected -- it drove
    # recall to 1.0 but tanked precision to ~0.26 (nearly everything got
    # flagged as captured). sqrt tempers that while still lifting recall.
    n_neg = (ytr_cap.values == 0).sum()
    n_pos = (ytr_cap.values == 1).sum()
    pos_weight = float(np.sqrt(n_neg / max(n_pos, 1)))

    def cap_sample_weight(y):
        return np.where(y == 1, pos_weight, 1.0)

    # ================= 4 single-task models =================
    set_seeds()
    print("Training rmin_model...")
    rmin_model = train_single(make_regressor(), X_tr_s, sc(rmin_scaler, ytr_rmin),
                               X_val_s, sc(rmin_scaler, yval_rmin), "mse", ["mae"])
    pred = rmin_scaler.inverse_transform(rmin_model.predict(X_te_s, verbose=0))
    print("  test R2:", r2_score(yte_rmin, pred))
    rmin_model.save(os.path.join(out_root, "models", "rmin_model.keras"))

    print("Training phi_model...")
    set_seeds()
    phi_model = train_single(make_regressor(), X_tr_s, sc_log(phi_scaler, ytr_phi),
                              X_val_s, sc_log(phi_scaler, yval_phi), "mse", ["mae"])
    pred = inv_log(phi_scaler, phi_model.predict(X_te_s, verbose=0))
    print("  test R2:", r2_score(yte_phi, pred))
    phi_model.save(os.path.join(out_root, "models", "phi_model.keras"))

    print("Training alpha_model (deflection_angle)...")
    set_seeds()
    alpha_model = train_single(make_regressor(), X_tr_s, sc_log(alpha_scaler, ytr_alpha),
                                X_val_s, sc_log(alpha_scaler, yval_alpha), "mse", ["mae"])
    pred = inv_log(alpha_scaler, alpha_model.predict(X_te_s, verbose=0))
    print("  test R2:", r2_score(yte_alpha, pred))
    alpha_model.save(os.path.join(out_root, "models", "alpha_model.keras"))

    print("Training capture_model...")
    set_seeds()
    capture_model = train_single(make_classifier(), X_tr_s, ytr_cap.values,
                                  X_val_s, yval_cap.values, "binary_crossentropy",
                                  ["accuracy"],
                                  sample_weight=cap_sample_weight(ytr_cap.values),
                                  val_sample_weight=cap_sample_weight(yval_cap.values))
    pred = (capture_model.predict(X_te_s, verbose=0) > 0.5).astype(int)
    print(classification_report(yte_cap, pred))
    capture_model.save(os.path.join(out_root, "models", "capture_model.keras"))

    # ================= multitask model =================
    print("Training multitask_model...")
    set_seeds()
    mt = build_multitask_model(cap_pos_weight=pos_weight)
    es = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)

    mt.fit(
        X_tr_s,
        {"rmin": sc(rmin_scaler, ytr_rmin), "phi": sc_log(phi_scaler, ytr_phi),
         "alpha": sc_log(alpha_scaler, ytr_alpha), "captured": ytr_cap.values},
        validation_data=(
            X_val_s,
            {"rmin": sc(rmin_scaler, yval_rmin), "phi": sc_log(phi_scaler, yval_phi),
             "alpha": sc_log(alpha_scaler, yval_alpha), "captured": yval_cap.values},
        ),
        epochs=150, batch_size=32, callbacks=[es], verbose=0,
    )
    preds = mt.predict(X_te_s, verbose=0)
    rmin_p, phi_p, alpha_p, cap_p = preds
    print("  rmin R2:", r2_score(yte_rmin, rmin_scaler.inverse_transform(rmin_p)))
    print("  phi  R2:", r2_score(yte_phi, inv_log(phi_scaler, phi_p)))
    print("  alpha R2:", r2_score(yte_alpha, inv_log(alpha_scaler, alpha_p)))
    print(classification_report(yte_cap, (cap_p > 0.5).astype(int)))
    mt.save(os.path.join(out_root, "models", "multitask_model.keras"))

    print(f"\nDone. Project saved under: {out_root}")


main(
    csv_path="/kaggle/input/datasets/vagishayadav166/sgr-photon-dataset/sgr_photon_dataset.csv",
    out_root="/kaggle/working/project"
)


Training rmin_model...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


  test R2: 0.9999987516580349
Training phi_model...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


  test R2: 0.9912973532789167
Training alpha_model (deflection_angle)...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


  test R2: 0.9963616622660398
Training capture_model...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


              precision    recall  f1-score   support

           0       1.00      0.93      0.96      1043
           1       0.48      1.00      0.65        69

    accuracy                           0.93      1112
   macro avg       0.74      0.96      0.81      1112
weighted avg       0.97      0.93      0.94      1112

Training multitask_model...
  rmin R2: 0.9999010170292615
  phi  R2: 0.992872807454795
  alpha R2: 0.9955208809601992
              precision    recall  f1-score   support

           0       1.00      0.93      0.96      1043
           1       0.48      1.00      0.65        69

    accuracy                           0.93      1112
   macro avg       0.74      0.96      0.81      1112
weighted avg       0.97      0.93      0.94      1112


Done. Project saved under: /kaggle/working/project
